In [1]:
import numpy as np
import pandas as pd

In [3]:
dataset = pd.read_csv('datasets/kpi_with_congestion.csv')
dataset_cell_info = pd.read_csv('datasets/final_info_data.csv')

C:\Users\yassi\AppData\Local\Temp\ipykernel_21692\146474988.py:1: DtypeWarning: Columns (0: CongestionLevelHour, 1: BusyHour_Level, 2: CongestionLevel) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset = pd.read_csv('datasets/kpi_with_congestion.csv')


In [4]:
dataset_cell_info.head()

,Cell Name,Longitude_Sector,Latitude_Sector,CoverageType,Azimuth,LAC,expected_cell_range
0,ARI_0003_C01_309_f1,10.16641,36.89404,outdoor,160.0,10016,630.0
1,ARI_0003_C01_309_f2,10.16641,36.89404,outdoor,250.0,10016,357.0
2,ARI_0003_C01_309_f3,10.16641,36.89404,outdoor,340.0,10016,NaN
3,ARI_0003_C01_309_f4,10.16641,36.89404,outdoor,70.0,10016,763.0
4,ARI_0003_C01_309_h1,10.16641,36.89404,outdoor,160.0,10016,630.0


In [5]:
dataset_cell_info["Cell Name"].unique()

<StringArray>
['ARI_0003_C01_309_f1', 'ARI_0003_C01_309_f2', 'ARI_0003_C01_309_f3',
 'ARI_0003_C01_309_f4', 'ARI_0003_C01_309_h1', 'ARI_0003_C01_309_h2',
 'ARI_0003_C01_309_h3', 'ARI_0003_C01_309_h4', 'ARI_0003_C01_309_l1',
 'ARI_0003_C01_309_l2',
 ...
 'ZAG_0158_C01_125_f3', 'ZAG_0158_C01_125_f4', 'ZAG_0158_C01_125_h1',
 'ZAG_0158_C01_125_h2', 'ZAG_0158_C01_125_h3', 'ZAG_0158_C01_125_h4',
 'ZAG_0158_C01_125_l1', 'ZAG_0158_C01_125_l2', 'ZAG_0158_C01_125_l3',
 'ZAG_0158_C01_125_l4']
Length: 25734, dtype: str

In [ ]:
extracted = (
            dataset_cell_info["Cell Name"]
            .astype(str)
            .str.strip()
            .str.extract(r"^([A-Za-z]{3})_(\d+)_([A-Za-z0-9]+)$")
        )
region = extracted[0].fillna(
    dataset_cell_info["Cell Name"].astype(str).str.extract(r"^([A-Za-z]{3})")[0]
).str.upper()
dataset_cell_info["Region"] = region
dataset_cell_info["Radio Site"] = region + "_" + extracted[1].fillna(dataset_cell_info["Cell Name"].astype(str).str.extract(r"^[A-Za-z]{3}_(\d+)")[0])


In [6]:
dataset_cell_info["cell_id"] = dataset_cell_info["Cell Name"].str.extract(r'_([^_]+)$')

In [7]:
dataset_cell_info["cell_id"].unique()

<StringArray>
[   'f1',    'f2',    'f3',    'f4',    'h1',    'h2',    'h3',    'h4',
    'l1',    'l2',    'l3',    'l4',    'j1',    'j2',    'j3',    'g1',
    'g2',    'g3',    'k1',    'k2',    'k3',   'hx3',   'lx3',   'hx2',
   'lx2',   'hx1',   'lx1',    'K3',    'j4',    'h5',    'l5',    'l7',
    'g4',    'k4',   'hx4',   'lx4',    'f5',    'f6',    'h6',   'jx4',
   'jx1',   'jx2',   'jx3',    'L4', '217l1', '217l2', '217l3',    'f8',
    'h8']
Length: 49, dtype: str

In [8]:
dataset_cell_info["cell_id"].value_counts()

cell_id
f1       2369
f2       2367
f3       2319
h1       2205
h2       2185
h3       2141
l1       1814
l2       1808
l3       1772
f4        656
h4        625
l4        545
j1        538
j2        525
g1        513
j3        501
g2        484
g3        483
k2        443
k1        433
k3        420
j4        161
hx2        61
lx2        60
hx3        46
lx3        46
hx1        43
lx1        43
g4         19
k4         17
jx2        16
jx1        15
jx3        13
hx4        11
lx4        11
jx4         5
f5          4
h5          3
l5          2
f6          2
h6          2
K3          1
l7          1
L4          1
217l1       1
217l2       1
217l3       1
f8          1
h8          1
Name: count, dtype: int64

In [11]:
import re

def get_frequency(cell_id):
    cell_id = str(cell_id).lower()
    # strip any leading digits (handles the 217l1 error case)
    cell_id = re.sub(r'^\d+', '', cell_id)
    # grab the first letter (ignores 'x' since it comes after, and digits)
    match = re.match(r'^([a-z])', cell_id)
    if not match:
        return 'Unknown'
    
    letter = match.group(1)
    
    frequency_map = {
        'h': '1800',
        'l': '2100',
        'f': '800',
        'j': '800',
        'g': 'TDD',
        'k': 'TDD',
    }
    return frequency_map.get(letter, 'Unknown')

dataset_cell_info['frequency'] = dataset_cell_info['cell_id'].apply(get_frequency)

In [12]:
print(dataset_cell_info.groupby('frequency').size())
# or to verify the mapping is sensible:
print(dataset_cell_info.groupby(['frequency', 'cell_id']).size())

frequency
1800    7323
2100    6106
800     9492
TDD     2813
dtype: int64
frequency  cell_id
1800       h1         2205
           h2         2185
           h3         2141
           h4          625
           h5            3
           h6            2
           h8            1
           hx1          43
           hx2          61
           hx3          46
           hx4          11
2100       217l1         1
           217l2         1
           217l3         1
           L4            1
           l1         1814
           l2         1808
           l3         1772
           l4          545
           l5            2
           l7            1
           lx1          43
           lx2          60
           lx3          46
           lx4          11
800        f1         2369
           f2         2367
           f3         2319
           f4          656
           f5            4
           f6            2
           f8            1
           j1          538
           j2  

In [15]:
dataset_cell_info.head()

,Cell Name,Longitude_Sector,Latitude_Sector,CoverageType,Azimuth,LAC,expected_cell_range,frequency
0,ARI_0003_C01_309_f1,10.16641,36.89404,outdoor,160.0,10016,630.0,800
1,ARI_0003_C01_309_f2,10.16641,36.89404,outdoor,250.0,10016,357.0,800
2,ARI_0003_C01_309_f3,10.16641,36.89404,outdoor,340.0,10016,NaN,800
3,ARI_0003_C01_309_f4,10.16641,36.89404,outdoor,70.0,10016,763.0,800
4,ARI_0003_C01_309_h1,10.16641,36.89404,outdoor,160.0,10016,630.0,1800


In [17]:
dataset_cell_info.info()

<class 'pandas.DataFrame'>
RangeIndex: 25734 entries, 0 to 25733
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Cell Name            25734 non-null  str    
 1   Longitude_Sector     25734 non-null  float64
 2   Latitude_Sector      25734 non-null  float64
 3   CoverageType         25734 non-null  str    
 4   Azimuth              25734 non-null  float64
 5   LAC                  25734 non-null  int64  
 6   expected_cell_range  16420 non-null  float64
 7   frequency            25734 non-null  str    
dtypes: float64(4), int64(1), str(3)
memory usage: 1.6 MB


In [14]:
dataset_cell_info.drop(columns=['cell_id'], inplace=True)

In [18]:
dataset_cell_info.to_csv('datasets/final_info_data.csv', index=False)